# Candle Prediction using Market Depth

In [903]:
from pathlib import Path
import pandas as pd
import numpy as np
from strategies_dev.utils import resample_fractional_minute, apply_trailing_logic, generate_signal

In [904]:
# ---- Input ------
date_ = "23APR2026"
file_name = "NIFTY26APR24200PE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"

    # Volume computation
    # 1. Calculate the basic difference between rows
    df["volume_at_tick"] = df["volume_traded"].diff()
    df.loc[df["volume_at_tick"] == 0, "volume_at_tick"] = np.nan
    # df["volume_at_tick"] = df["volume_at_tick"].ffill()
    # df["volume_at_tick"] = df["volume_at_tick"].fillna(0)
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])
target_date = pd.to_datetime(date_).date()
df = df[df[col_name].dt.date == target_date]

In [905]:
file_name

'NIFTY26APR24200PE.xlsx'

In [906]:
df[["last_trade_time", "last_price", "depth", "volume_traded", "volume_at_tick"]].head(2)

,last_trade_time,last_price,depth,volume_traded,volume_at_tick
2,2026-04-23 09:15:00,226.05,"{'buy': [{'quantity': 455, 'price': 222.25, 'o...",24765,24765.0
3,2026-04-23 09:15:00,226.05,"{'buy': [{'quantity': 455, 'price': 222.25, 'o...",24765,NaN


In [907]:
# ---- Input ------
N = 6
window = 2
price_pct_threshold = 0.001
volume_threshold=1000
volume_period = 30

use_price_pct_level=True
use_price_trend=True
use_volume=False

In [908]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick'],
      dtype='str')

In [909]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-23 09:15:00.619000
2026-04-23 09:15:00


In [910]:
clubbed_df = resample_fractional_minute(df, col_name, N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["price_diff"] = clubbed_df["close"] - clubbed_df["open"]
clubbed_df["price_pct"] = (clubbed_df["close"] - clubbed_df["open"])/clubbed_df["open"]

clubbed_df["volume_diff"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_participated"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_pct"] = (clubbed_df["volume_close"] - clubbed_df["volume_open"])/clubbed_df["volume_open"]
clubbed_df["volume_avg"] = clubbed_df["volume_close"].rolling(volume_period).median()

In [911]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].head()

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,minute_close,depth_list,ltp_list,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg
270,2026-04-23 10:00:00,2026-04-23 10:00:00,175.20,176.15,174.45,175.05,10140.0,10595.0,2470.0,10595.0,...,178.1,"[{'buy': [{'quantity': 1170, 'price': 174.6, '...","[175.2, 174.9, 175.0, 176.0, 175.0, 176.1, 176...",2026-04-23 10:00:10,-0.15,-0.000856,8125.0,8125.0,0.044872,6792.5
271,2026-04-23 10:00:10,2026-04-23 10:00:00,174.65,175.35,174.60,174.90,2730.0,6175.0,1365.0,1365.0,...,178.1,"[{'buy': [{'quantity': 1170, 'price': 174.6, '...","[175.2, 174.9, 175.0, 176.0, 175.0, 176.1, 176...",2026-04-23 10:00:20,0.25,0.001431,4810.0,4810.0,-0.500000,6792.5
272,2026-04-23 10:00:20,2026-04-23 10:00:00,174.95,175.25,174.50,175.15,6695.0,6695.0,1885.0,5395.0,...,178.1,"[{'buy': [{'quantity': 1170, 'price': 174.6, '...","[175.2, 174.9, 175.0, 176.0, 175.0, 176.1, 176...",2026-04-23 10:00:30,0.20,0.001143,4810.0,4810.0,-0.194175,6662.5
273,2026-04-23 10:00:30,2026-04-23 10:00:00,175.60,177.00,174.90,176.75,2015.0,6695.0,1105.0,6695.0,...,178.1,"[{'buy': [{'quantity': 1170, 'price': 174.6, '...","[175.2, 174.9, 175.0, 176.0, 175.0, 176.1, 176...",2026-04-23 10:00:40,1.15,0.006549,5590.0,5590.0,2.322581,6630.0
274,2026-04-23 10:00:40,2026-04-23 10:00:00,176.80,178.65,176.75,178.65,4485.0,20215.0,3445.0,9425.0,...,178.1,"[{'buy': [{'quantity': 1170, 'price': 174.6, '...","[175.2, 174.9, 175.0, 176.0, 175.0, 176.1, 176...",2026-04-23 10:00:50,1.85,0.010464,16770.0,16770.0,1.101449,6727.5


In [912]:
clubbed_df[["volume_open", "volume_high", "volume_low", "volume_close", "volume_participated"]]

,volume_open,volume_high,volume_low,volume_close,volume_participated
0,24765.0,168285.0,24765.0,87555.0,143520.0
1,94315.0,101075.0,48100.0,48100.0,52975.0
2,52585.0,80730.0,38025.0,39585.0,42705.0
3,30485.0,36725.0,29965.0,35815.0,6760.0
4,52455.0,52455.0,19045.0,25805.0,33410.0
...,...,...,...,...,...
2245,10985.0,17290.0,10660.0,15795.0,6630.0
2246,7670.0,13585.0,7670.0,10530.0,5915.0
2247,21125.0,24960.0,9815.0,9815.0,15145.0
2248,7020.0,12415.0,4225.0,12415.0,8190.0


In [913]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].shape

(36, 24)

In [914]:
# generate_signal
clubbed_df_2 = generate_signal(clubbed_df, window, price_pct_threshold = price_pct_threshold, volume_threshold=volume_threshold,
                               use_price_pct_level=use_price_pct_level, use_price_trend=use_price_trend, use_volume=use_volume)
chk=1

use_price_pct_level :  True
use_price_trend :  True
use_volume :  False


In [915]:
def evaluate_signals(df, forward_candles=3):
    """
    For each BUY/SELL signal, check what price did
    over the next `forward_candles` bars.
    """
    df = df.copy()

    # Forward return: % change from signal close to N candles later
    df["fwd_return"] = df["close"].shift(-forward_candles) / df["close"] - 1

    buy_signals  = df[df["predicted"] == "BUY"]
    sell_signals = df[df["predicted"] == "SELL"]

    print(f"\n=== SIGNAL QUALITY  (forward {forward_candles} candles) ===")

    if len(buy_signals):
        buy_correct  = (buy_signals["fwd_return"] > 0).sum()
        buy_accuracy = buy_correct / len(buy_signals) * 100
        buy_avg_ret  = buy_signals["fwd_return"].mean() * 100
        print(f"BUY  signals : {len(buy_signals):>4}  |  "
              f"accuracy: {buy_accuracy:.1f}%  |  "
              f"avg fwd return: {buy_avg_ret:+.4f}%")

    if len(sell_signals):
        sell_correct  = (sell_signals["fwd_return"] < 0).sum()
        sell_accuracy = sell_correct / len(sell_signals) * 100
        sell_avg_ret  = sell_signals["fwd_return"].mean() * 100
        print(f"SELL signals : {len(sell_signals):>4}  |  "
              f"accuracy: {sell_accuracy:.1f}%  |  "
              f"avg fwd return: {sell_avg_ret:+.4f}%")

    # Distribution of forward returns
    print(f"\nBUY  fwd return distribution:\n"
          f"{buy_signals['fwd_return'].describe().apply(lambda x: f'{x*100:+.4f}%')}")
    print(f"\nSELL fwd return distribution:\n"
          f"{sell_signals['fwd_return'].describe().apply(lambda x: f'{x*100:+.4f}%')}")

    return df


df_ = evaluate_signals(clubbed_df, forward_candles=3)


=== SIGNAL QUALITY  (forward 3 candles) ===
BUY  signals :   92  |  accuracy: 44.6%  |  avg fwd return: -0.1241%
SELL signals :   65  |  accuracy: 35.4%  |  avg fwd return: +0.3013%

BUY  fwd return distribution:
count    +9200.0000%
mean        -0.1241%
std         +1.3244%
min         -3.3799%
25%         -1.0175%
50%         -0.1790%
75%         +0.7140%
max         +3.0073%
Name: fwd_return, dtype: str

SELL fwd return distribution:
count    +6500.0000%
mean        +0.3013%
std         +1.2156%
min         -2.5811%
25%         -0.3351%
50%         +0.2970%
75%         +0.8888%
max         +3.9705%
Name: fwd_return, dtype: str


In [916]:
df_

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,ltp_list,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg,predicted,fwd_return
0,2026-04-23 09:15:00,2026-04-23 09:15:00,226.05,255.40,226.05,247.95,24765.0,168285.0,24765.0,87555.0,...,"[226.05, 226.05, 234.15, 234.15, 239.0, 239.0,...",2026-04-23 09:15:10,21.90,0.096881,143520.0,143520.0,2.535433,NaN,NaN,-0.020770
1,2026-04-23 09:15:10,2026-04-23 09:15:00,247.90,252.30,244.50,248.80,94315.0,101075.0,48100.0,48100.0,...,"[226.05, 226.05, 234.15, 234.15, 239.0, 239.0,...",2026-04-23 09:15:20,0.90,0.003630,52975.0,52975.0,-0.490007,NaN,NaN,-0.024518
2,2026-04-23 09:15:20,2026-04-23 09:15:00,247.95,250.35,246.50,250.35,52585.0,80730.0,38025.0,39585.0,...,"[226.05, 226.05, 234.15, 234.15, 239.0, 239.0,...",2026-04-23 09:15:30,2.40,0.009679,42705.0,42705.0,-0.247219,NaN,BUY,-0.012582
3,2026-04-23 09:15:30,2026-04-23 09:15:00,250.45,250.45,242.80,242.80,30485.0,36725.0,29965.0,35815.0,...,"[226.05, 226.05, 234.15, 234.15, 239.0, 239.0,...",2026-04-23 09:15:40,-7.65,-0.030545,6760.0,6760.0,0.174840,NaN,NaN,0.031301
4,2026-04-23 09:15:40,2026-04-23 09:15:00,245.25,245.25,241.95,242.70,52455.0,52455.0,19045.0,25805.0,...,"[226.05, 226.05, 234.15, 234.15, 239.0, 239.0,...",2026-04-23 09:15:50,-2.55,-0.010398,33410.0,33410.0,-0.508055,NaN,NaN,0.037495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2245,2026-04-23 15:29:10,2026-04-23 15:29:00,203.55,203.90,203.00,203.90,10985.0,17290.0,10660.0,15795.0,...,"[203.35, 203.35, 204.0, 204.0, 203.85, 203.85,...",2026-04-23 15:29:20,0.35,0.001719,6630.0,6630.0,0.437870,6240.0,NaN,0.000981
2246,2026-04-23 15:29:20,2026-04-23 15:29:00,204.00,204.00,203.10,203.90,7670.0,13585.0,7670.0,10530.0,...,"[203.35, 203.35, 204.0, 204.0, 203.85, 203.85,...",2026-04-23 15:29:30,-0.10,-0.000490,5915.0,5915.0,0.372881,6630.0,NaN,0.001962
2247,2026-04-23 15:29:30,2026-04-23 15:29:00,203.85,203.95,203.00,203.45,21125.0,24960.0,9815.0,9815.0,...,"[203.35, 203.35, 204.0, 204.0, 203.85, 203.85,...",2026-04-23 15:29:40,-0.40,-0.001962,15145.0,15145.0,-0.535385,7507.5,NaN,NaN
2248,2026-04-23 15:29:40,2026-04-23 15:29:00,203.70,204.20,203.60,204.10,7020.0,12415.0,4225.0,12415.0,...,"[203.35, 203.35, 204.0, 204.0, 203.85, 203.85,...",2026-04-23 15:29:50,0.40,0.001964,8190.0,8190.0,0.768519,8287.5,NaN,NaN


In [917]:
# clubbed_df_2[8:11].to_excel("clubbed_df_8_11.xlsx")

In [918]:
clubbed_df_2.columns

Index(['bucket_time', 'minute', 'open', 'high', 'low', 'close', 'volume_open',
       'volume_high', 'volume_low', 'volume_close', 'candle_type',
       'minute_open', 'minute_high', 'minute_low', 'minute_close',
       'depth_list', 'ltp_list', 'bucket_time_next', 'price_diff', 'price_pct',
       'volume_diff', 'volume_participated', 'volume_pct', 'volume_avg',
       'predicted'],
      dtype='str')

In [919]:
# clubbed_df_2[["bucket_time", "bucket_time_next", "price_pct", "price_diff", "predicted"]].head(40)

In [920]:
clubbed_df_2[clubbed_df_2["predicted"]==clubbed_df["candle_type"]].tail(5)

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,depth_list,ltp_list,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg,predicted
2154,2026-04-23 15:14:00,2026-04-23 15:14:00,217.1,220.00,217.10,220.00,22165.0,87685.0,11245.0,11245.0,...,"[{'buy': [{'quantity': 325, 'price': 216.7, 'o...","[217.1, 217.1, 218.15, 218.15, 218.9, 218.9, 2...",2026-04-23 15:14:10,2.90,0.013358,76440.0,76440.0,-0.492669,8775.0,BUY
2174,2026-04-23 15:17:20,2026-04-23 15:17:00,211.9,211.90,208.65,209.00,15795.0,23400.0,12610.0,14820.0,...,"[{'buy': [{'quantity': 1105, 'price': 216.0, '...","[216.05, 216.05, 215.85, 215.85, 215.25, 215.2...",2026-04-23 15:17:30,-2.90,-0.013686,10790.0,10790.0,-0.061728,11635.0,SELL
2194,2026-04-23 15:20:40,2026-04-23 15:20:00,204.3,204.40,202.05,202.05,5915.0,15730.0,5915.0,7800.0,...,"[{'buy': [{'quantity': 715, 'price': 208.25, '...","[208.25, 208.25, 208.35, 208.35, 208.0, 208.0,...",2026-04-23 15:20:50,-2.25,-0.011013,9815.0,9815.0,0.318681,6760.0,SELL
2203,2026-04-23 15:22:10,2026-04-23 15:22:00,202.0,203.25,201.95,203.25,4875.0,16575.0,3315.0,5655.0,...,"[{'buy': [{'quantity': 2275, 'price': 200.35, ...","[200.05, 200.05, 200.35, 200.35, 200.4, 200.4,...",2026-04-23 15:22:20,1.25,0.006188,13260.0,13260.0,0.160000,5200.0,BUY
2206,2026-04-23 15:22:40,2026-04-23 15:22:00,205.2,205.75,204.80,205.70,2990.0,4420.0,1495.0,2535.0,...,"[{'buy': [{'quantity': 2275, 'price': 200.35, ...","[200.05, 200.05, 200.35, 200.35, 200.4, 200.4,...",2026-04-23 15:22:50,0.50,0.002437,2925.0,2925.0,-0.152174,4972.5,BUY


In [921]:
clubbed_df_2.shape

(2250, 25)

In [922]:
clubbed_df_2 = clubbed_df[:-1]

In [923]:
clubbed_df_2.shape

(2249, 25)

In [924]:
# clubbed_df.to_excel("clubbed_df.xlsx")

In [925]:
# df_with_signal = df.merge(
#         clubbed_df[["bucket_time", "predicted"]],
#         left_on="last_trade_time",
#         right_on="bucket_time",
#         how="left"
#     )

# import pandas as pd

# 1. Ensure both DataFrames are sorted by the time columns
df = df.sort_values("last_trade_time")
clubbed_df_2 = clubbed_df_2.sort_values("bucket_time")

# 2. Perform the proximity merge
df_with_signal = pd.merge_asof(
    df,
    clubbed_df_2[["bucket_time_next", "predicted"]],
    left_on="last_trade_time",
    right_on="bucket_time_next",
    direction="backward" # Only looks at the past/current, never the future
)

# Find duplicates in bucket_time and set their 'predicted' value to NaN
df_with_signal.loc[df_with_signal.duplicated(subset=['bucket_time_next'], keep='first'), 'predicted'] = np.nan

In [926]:
# df_with_signal.to_excel("df_with_signal.xlsx")

In [927]:
# clubbed_df_2.to_excel("clubbed_df_2.xlsx")

In [928]:
df_with_signal.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_at_tick,bucket_time_next,predicted
0,18501634,NIFTY26APR24200PE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,226.05,130,221.61,PE,atm,...,53.514431,2532985,2532985,2532985,"{'buy': [{'quantity': 455, 'price': 222.25, 'o...",True,full,24765.0,NaT,NaN
1,18501634,NIFTY26APR24200PE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,226.05,130,221.61,PE,atm,...,53.514431,2532985,2532985,2532985,"{'buy': [{'quantity': 455, 'price': 222.25, 'o...",True,full,NaN,NaT,NaN
2,18501634,NIFTY26APR24200PE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,234.15,65,221.61,PE,atm,...,59.015280,2532985,2532985,2532985,"{'buy': [{'quantity': 455, 'price': 222.25, 'o...",True,full,NaN,NaT,NaN
3,18501634,NIFTY26APR24200PE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,234.15,65,221.61,PE,atm,...,59.015280,2532985,2532985,2532985,"{'buy': [{'quantity': 455, 'price': 222.25, 'o...",True,full,NaN,NaT,NaN
4,18501634,NIFTY26APR24200PE,NaN,2026-04-23 09:15:02.120,2026-04-23 09:15:01,239.00,65,231.90,PE,atm,...,62.308998,2532985,2532985,2532985,"{'buy': [{'quantity': 455, 'price': 237.4, 'or...",True,full,145145.0,NaT,NaN


In [929]:
df_with_signal.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick', 'bucket_time_next', 'predicted'],
      dtype='str')

In [930]:
# params = {
#     "initial_sl_pct": 0.02,
#     "target_pct": 0.01,
#     "trail_sl_pct": 0.02,
#     "tight_sl_offset": 0.5,
# }

params = {
    "initial_sl_pct": 0.02,
    "target_pct": 0.001,
    "trail_sl_pct": 0.018,
    "tight_sl_offset": 0.5,
}

In [931]:
trades = apply_trailing_logic(df_with_signal, params)

trades = pd.DataFrame(trades)
if len(trades):
    trades["final"] = trades.apply(lambda row: "profit" if row["profit"] > 0 else "loss", axis=1)
else:
    print("trades not generated")

In [932]:
len(trades)

91

In [933]:
# trades

## Profit

In [934]:
trades[trades["final"]=="profit"]["profit"].sum(), trades[trades["final"]=="profit"]["pnl"].sum()

(np.float64(57.549999999999926), np.float64(21384.72272034988))

### Loss

In [935]:
trades[trades["final"]=="loss"]["profit"].sum(), trades[trades["final"]=="loss"]["pnl"].sum()

(np.float64(-30.220700000000136), np.float64(-28961.825330566156))

### Final

In [936]:
trades[trades["final"]=="profit"]["pnl"].sum() + trades[trades["final"]=="loss"]["pnl"].sum()

np.float64(-7577.102610216276)

In [937]:
trades.head(11)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,pnl,final
0,2026-04-23 09:15:31,250.45,2026-04-23 09:15:38,245.9419,-4.5081,-0.018000,-3264.448896,loss
1,2026-04-23 09:22:10,218.25,2026-04-23 09:22:11,220.3500,2.1000,0.009622,1063.298683,profit
2,2026-04-23 09:34:00,218.80,2026-04-23 09:34:02,218.9000,0.1000,0.000457,-235.874243,profit
3,2026-04-23 09:36:10,220.35,2026-04-23 09:36:16,221.5000,1.1500,0.005219,444.060333,profit
4,2026-04-23 09:36:30,224.00,2026-04-23 09:36:30,223.8500,-0.1500,-0.000670,-404.218254,loss
5,2026-04-23 09:38:10,220.40,2026-04-23 09:38:11,220.8000,0.4000,0.001815,-42.948412,profit
6,2026-04-23 09:40:31,214.40,2026-04-23 09:40:36,215.5000,1.1000,0.005131,418.493318,profit
7,2026-04-23 09:40:40,216.05,2026-04-23 09:40:40,216.1000,0.0500,0.000231,-265.150231,profit
8,2026-04-23 09:45:11,200.20,2026-04-23 09:45:13,200.1000,-0.1000,-0.000500,-344.169383,loss
9,2026-04-23 09:55:00,179.25,2026-04-23 09:55:03,176.0235,-3.2265,-0.018000,-2349.822738,loss


In [938]:
trades["final"].value_counts()

final
profit    57
loss      34
Name: count, dtype: int64

In [939]:
trades["final"].value_counts(normalize=True) * 100

final
profit    62.637363
loss      37.362637
Name: proportion, dtype: float64

In [940]:
trades[trades["final"]=="loss"].head(12)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,pnl,final
0,2026-04-23 09:15:31,250.45,2026-04-23 09:15:38,245.9419,-4.5081,-0.018000,-3264.448896,loss
4,2026-04-23 09:36:30,224.00,2026-04-23 09:36:30,223.8500,-0.1500,-0.000670,-404.218254,loss
8,2026-04-23 09:45:11,200.20,2026-04-23 09:45:13,200.1000,-0.1000,-0.000500,-344.169383,loss
9,2026-04-23 09:55:00,179.25,2026-04-23 09:55:03,176.0235,-3.2265,-0.018000,-2349.822738,loss
10,2026-04-23 09:56:40,177.35,2026-04-23 09:56:41,177.2000,-0.1500,-0.000846,-350.148385,loss
12,2026-04-23 10:00:51,178.60,2026-04-23 10:00:56,178.5500,-0.0500,-0.000280,-286.670431,loss
15,2026-04-23 10:15:41,180.70,2026-04-23 10:15:43,180.5000,-0.2000,-0.001107,-386.494602,loss
16,2026-04-23 10:21:00,173.25,2026-04-23 10:21:00,173.1000,-0.1500,-0.000866,-345.396264,loss
18,2026-04-23 10:25:10,179.00,2026-04-23 10:25:12,178.7500,-0.2500,-0.001397,-416.987597,loss
19,2026-04-23 10:34:10,193.75,2026-04-23 10:34:11,193.5500,-0.2000,-0.001032,-401.620257,loss


In [941]:
trades[trades["final"]=="loss"].columns

Index(['entry_time', 'entry_price', 'exit_time', 'exit_price', 'profit',
       'profit_pct', 'pnl', 'final'],
      dtype='str')

In [942]:

# clubbed_df_2.to_excel(Path(f"assets/logs/{date_}/clubbed_df.xlsx"))

In [943]:
# clubbed_df_2["predicted"].value_counts()